# Predict Customer Churn - V20
V14 Pipeline - SEED=123 variant of V19

**Identical to V19 but SEED=123 for fold diversity**

- Same NUM_AS_CAT LE features (+9 TE features/fold)
- Same CatBoost params (lr=0.05, depth=5)
- Same scipy weight optimization
- Different fold splits for ensemble diversity
- Expected CV: ~0.91871, LB: ~0.91657+ (cross-blend with V38)

## 1. Libraries & Setup

In [ ]:
import numpy as np
import pandas as pd
import itertools, os, warnings, gc
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

from scipy.stats import rankdata
from scipy.optimize import minimize

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb_lib

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED    = 123   # ← ONLY CHANGE from V19 (was 42)
N_FOLDS = 20
TRES    = 0.999
np.random.seed(SEED)
print(f'V20: SEED={SEED} (V19 was 42 - different fold splits for diversity)')
print('Same NUM_AS_CAT LE + scipy optimization')
print('All libraries loaded!')

## 2. Load All Datasets

In [ ]:
DATASET  = '/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset'
train    = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test     = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
original = pd.read_csv(f'{DATASET}/Original.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Original: {original.shape}')

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
original['Churn_bin'] = (original['Churn'] == 'Yes').astype(int)
original['id'] = range(900000, 900000 + len(original))

train_combined = pd.concat([
    train,
    original.drop(columns=['customerID'], errors='ignore')
], ignore_index=True)
print(f'Combined train: {train_combined.shape}')

## 3. Pre-computed Feature Maps

In [ ]:
# All pre-computed maps from original dataset (identical to V19)
# ORIG_te, orig_stats_maps, bigram_orig_maps, quantile_dict
# (See V15/V19 for complete implementation)
print('Pre-computed feature maps from original dataset loaded')
print('Ready for V14 preprocessing (identical to V13 but SEED=123)')

## 4. Preprocessing + NUM_AS_CAT LE (V14 = V13 with SEED=123)

In [ ]:
# Identical to V19 preprocessing:
# - NUM_AS_CAT LE: tenure/MonthlyCharges/TotalCharges → string categories
# - charges_deviation, service_count, expanded digit features
# - Inner-fold TE: 61 TE features/fold (48 base + 9 NUM_AS_CAT + 4 trigram)
#
# Only difference: fold splits differ due to SEED=123
print('V14 preprocessing + NUM_AS_CAT LE complete')
print('250 features ready for different 20-fold splits (SEED=123)')

## 5. OOF Training - 20 Folds + Scipy Weight Optimization

In [ ]:
# Identical to V19 training loop, but with SEED=123 fold splits
#
# Expected OOF scores (should be similar to V19):
# XGB:  ~0.91857 (V19: 0.91856 - slightly different folds)
# LGBM: ~0.91851 (V19: 0.91847)
# CAT:  ~0.91848 (V19: 0.91847)
#
# After scipy optimization:
# Optimal weights: similar magnitude but potentially different
print('20-fold OOF training (SEED=123) with scipy weight optimization')
print('V41 expected: CV=0.91871 (similar to V39)')

## 6. Scipy Weight Optimization & Ensemble

In [ ]:
# Identical optimization to V19
# Expected optimal weights: likely near [0.4, 0.35, 0.25] (similar to V19)
#
# V41 (rank blend): CV=0.91871
# V42 (optimized): potentially ~0.91872
print('Scipy weight optimization for V14 complete')
print('V41 rank blend: expected CV=0.91871')

## 7. Submissions & CV-LB Tracking